# Bài 3. Minimax và cắt tỉa Alpha-Beta

Trạng thái, nước hợp lệ và điểm kết thúc lấy từ `tictactoe.py`. Mỗi lần gọi tìm kiếm coi người sắp đi là Max, đối thủ là Min. Điểm chính xác là +1 nếu Max thắng, 0 nếu hòa và −1 nếu Max thua. Cả hai thuật toán tính giá trị bằng duyệt sâu. Alpha-Beta cập nhật cận dưới α ở Max và cận trên β ở Min, rồi bỏ qua các nhánh còn lại khi α ≥ β.

Một nút được tính ngay khi hàm đệ quy thăm nó, gồm gốc và lá. Hai phương pháp dùng cùng thứ tự tăng dần chỉ số ô và giữ nước đầu tiên nếu đồng điểm. Thời gian là một lần đo bằng `perf_counter()` trên mỗi trạng thái, tính bằng mili giây. Các phép kiểm tra điểm chính xác sau đó nằm ngoài đoạn đo.

Mỗi trạng thái sau có X tới lượt. Các ô được đánh số từ 0 đến 8 theo hàng.

| Trạng thái | Ô X | Ô O |
| --- | --- | --- |
| X thắng ngay | 0, 4 | 1, 3 |
| X phải chặn | 0, 2 | 1, 4 |
| X bất lợi (giả định) | 2 | 0, 8 |
| Bàn trống | ∅ | ∅ |
| X thua cưỡng bức | 3, 5, 6 | 0, 1, 4 |

In [1]:
from time import perf_counter
from tictactoe import other, result
from tictactoe_search import alpha_beta_search
from lab05_experiments import CASES, benchmark, game_value

def measure(search, board, player, **settings):
    start = perf_counter()
    record = search(board, player, **settings)
    return dict(record, ms=(perf_counter() - start) * 1000)

def exact_move_value(board, player, move):
    child = result(board, player, move)
    return -game_value(tuple(child), other(player))

def print_match(summary):
    for label, row in [*summary["by_role"].items(), ("Tổng", summary)]:
        print(
            f'{label:>4} {row["games"]:>4} {row["wins"]:>6} '
            f'{row["draws"]:>5} {row["losses"]:>5} {row["ms_per_move"]:>13.3f}'
        )

In [2]:
from tictactoe_search import minimax_search, alpha_beta_player, ordered_actions

results = []
for label, board, player in CASES:
    minimax = measure(minimax_search, board, player)
    alpha_beta = measure(alpha_beta_search, board, player)
    assert minimax["move"] == alpha_beta["move"]
    assert minimax["value"] == alpha_beta["value"]
    assert alpha_beta["nodes"] <= minimax["nodes"]
    assert exact_move_value(board, player, alpha_beta["move"]) == minimax["value"]
    results.append((label, minimax, alpha_beta))

print(f'{"Trạng thái":<28} {"Thuật toán":<12} {"Ô":>3} {"Điểm":>5} {"Nút":>9} {"ms":>10}')
for label, minimax, alpha_beta in results:
    for name, row in (("Minimax", minimax), ("Alpha-Beta", alpha_beta)):
        print(
            f'{label:<28} {name:<12} {row["move"]:>3} {row["value"]:>5} '
            f'{row["nodes"]:>9,} {row["ms"]:>10.3f}'
        )

Trạng thái                   Thuật toán     Ô  Điểm       Nút         ms
X thắng ngay                 Minimax        2     1       190      8.508
X thắng ngay                 Alpha-Beta     2     1        61      2.545
X phải chặn                  Minimax        7     0       206      8.573
X phải chặn                  Alpha-Beta     7     0       101      4.178
X bất lợi (giả định)         Minimax        1    -1       941     36.296
X bất lợi (giả định)         Alpha-Beta     1    -1       238      8.922
Bàn trống                    Minimax        0     0   549,946  28649.976
Bàn trống                    Alpha-Beta     0     0    18,297   1902.349
X thua cưỡng bức             Minimax        2    -1        10      0.846
X thua cưỡng bức             Alpha-Beta     2    -1         8      0.726


Minimax và Alpha-Beta trả cùng nước và cùng giá trị ở cả 5 trạng thái. Trong thế thắng, ô 8 kết thúc ván ngay, còn cả hai thuật toán chọn ô 2 vì nước này cũng bảo đảm thắng và được duyệt trước. Ở thế phải chặn, ô 7 giữ hòa. Với bàn trống, giá trị chính xác là 0 vì hai bên có thể giữ hòa khi chơi tối ưu. Minimax duyệt 549,946 nút ở bàn trống, còn Alpha-Beta duyệt 18,297 nút. Thế bất lợi từ notebook gốc giả định X tới lượt khi O có nhiều hơn X đúng một quân. Thế thua hợp lệ cho phép đối chiếu trên bàn có thể xuất hiện trong ván thật.

## Đổi thứ tự duyệt nước

Cấu hình thứ hai thử ô giữa trước, sau đó đến góc và cạnh. Mỗi nhóm dùng thứ tự chỉ số tăng dần. Hai cấu hình cùng tìm tới trạng thái kết thúc.

In [3]:
print("Thứ tự mới từ bàn trống:", ordered_actions([" "] * 9))
print(f'{"Trạng thái":<28} {"Nút gốc":>9} {"Nút mới":>9} {"Ô gốc":>6} {"Ô mới":>6} {"ms mới":>10}')
ordered_rows = []
for (label, board, player), (_, _, original) in zip(CASES, results):
    ordered = measure(alpha_beta_search, board, player, ordered=True)
    assert original["value"] == ordered["value"]
    assert exact_move_value(board, player, ordered["move"]) == original["value"]
    ordered_rows.append(ordered)
    print(
        f'{label:<28} {original["nodes"]:>9,} {ordered["nodes"]:>9,} '
        f'{original["move"]:>6} {ordered["move"]:>6} {ordered["ms"]:>10.3f}'
    )

Thứ tự mới từ bàn trống: [4, 0, 2, 6, 8, 1, 3, 5, 7]
Trạng thái                     Nút gốc   Nút mới  Ô gốc  Ô mới     ms mới
X thắng ngay                        61        44      2      2      4.155
X phải chặn                        101       149      7      7     12.566
X bất lợi (giả định)               238        51      1      4      4.097
Bàn trống                       18,297     7,275      0      4    551.773
X thua cưỡng bức                     8         8      2      2      0.483


Ở bàn trống, đổi thứ tự giảm số nút từ 18,297 xuống 7,275, tương đương 60.2%. Khi tìm thấy nước mạnh sớm, Max nâng α sớm hoặc Min hạ β sớm, nên gặp điều kiện cắt ở nhiều nhánh hơn. Với các trạng thái X phải chặn, cấu hình mới duyệt nhiều nút hơn. Nếu có nhiều nước cùng giá trị tối ưu, thay đổi thứ tự có thể đổi ô được chọn. Kiểm tra điểm của nước mới bằng tìm kiếm đầy đủ xác nhận nó vẫn tối ưu.

## Alpha-Beta đấu Random

Tác tử sử dụng thứ tự giữa–góc–cạnh. Nó chơi 100 ván ở vai X và 100 ván ở vai O với cùng giao thức ở các bài. Thời gian chỉ tính các lượt chọn nước của tác tử đang đo.

In [4]:
alpha_metrics = benchmark(alpha_beta_player)
print(f'{"Vai":>4} {"Ván":>4} {"Thắng":>6} {"Hòa":>5} {"Thua":>5} {"ms/nước":>13}')
print_match(alpha_metrics)
assert alpha_metrics["losses"] == 0

 Vai  Ván  Thắng   Hòa  Thua       ms/nước
   x  100     98     2     0       124.505
   o  100     87    13     0        45.136
Tổng  200    185    15     0        85.855


Alpha-Beta thắng 98 ván khi đi trước và 87 ván khi đi sau. Cả hai vai đều giữ được kết quả từ hòa trở lên như giá trị Minimax dự đoán.